# Skin Cancer Detection with EfficientNet-B0

**ISIC 2019 · PyTorch · Transfer Learning**

This notebook trains a 7-class skin lesion classifier on the ISIC 2019 dataset using EfficientNet-B0 with ImageNet pre-training. Each section below is annotated to explain the *why* behind every design decision.

---

## 1. Imports

We pull in the core libraries:
- **os / pandas / numpy / PIL** – file I/O, tabular data, and image loading.
- **torch & torchvision** – PyTorch for model building, transforms, and data loading.
- **sklearn** – group-aware train/val splitting and classification metrics.

`torch.backends.cudnn.benchmark = True` lets cuDNN auto-tune convolution algorithms for the fixed input size, giving a free speed boost on GPU.

In [1]:
import os
import pandas as pd
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report

torch.backends.cudnn.benchmark = True

## 2. Hyperparameters & Paths

Central config kept in one place so nothing is hard-coded elsewhere:
- **IMG_SIZE 224** – standard input resolution for EfficientNet (trained on ImageNet at this size).
- **BATCH_SIZE 32** – fits comfortably in GPU VRAM; increase if you have more memory.
- **EPOCHS 25** – enough for the cosine LR schedule to complete a full cycle.
- **LR 3e-4** – a reliable default for AdamW on fine-tuning tasks.
- **IMG_DIR** – points to shade-of-gray colour-corrected ISIC 2019 images on Kaggle.

In [2]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 25
NUM_WORKERS = 6
LR = 3e-4

IMG_DIR = "/kaggle/input/isic-2019-prepocessed/step3_shadesofgray"

## 3. Load & Merge Metadata + Ground Truth

ISIC 2019 ships two CSVs that must be joined:
1. `metadata.csv` – image IDs, lesion IDs, patient info.
2. `ISIC_2019_Training_GroundTruth.csv` – one-hot columns for 7 diagnostic classes.

**Label extraction:** `idxmax(axis=1)` on the one-hot columns returns the class name for each image. We keep only rows with exactly one positive label (`sum == 1`) to discard any ambiguous annotations.

In [3]:
meta = pd.read_csv("/kaggle/input/d/demirelmas/metadata-csv/metadata.csv")
gt   = pd.read_csv("/kaggle/input/isic-2019-training-groundtruth/ISIC_2019_Training_GroundTruth.csv")

df = meta.merge(gt, on="image", how="inner")

label_cols = ["MEL", "NV", "BCC", "DF", "AK", "BKL", "VASC"]

df = df.dropna().reset_index(drop=True)

df = df[df[label_cols].sum(axis=1) == 1].reset_index(drop=True)

df["label"] = df[label_cols].idxmax(axis=1)

## 4. Fill Missing Lesion IDs

Some images have no `lesion_id` (i.e. they are standalone, not part of a multi-image lesion). We fall back to the `image` column so that every row has a group key. This matters for the group-split in the next step.

In [4]:
df["lesion_id"] = df["lesion_id"].fillna(df["image"])

## 5. Class → Integer Mapping

Neural nets need integer targets, not strings. We sort the 7 class names alphabetically to get a deterministic ordering and build a simple dictionary. The `assert` confirms no label leaked through as NaN – a cheap sanity check that pays off later.

In [5]:
classes = sorted(df["label"].unique())
class_to_idx = {c: i for i, c in enumerate(classes)}

df["label_idx"] = df["label"].map(class_to_idx)

assert df["label_idx"].isna().sum() == 0

## 6. Group-Aware Train / Val Split (80 / 20)

`GroupShuffleSplit` ensures that **all images from the same lesion end up in the same split**. Without this, augmented or close-up views of the same lesion could appear in both train and val, causing inflated (leaked) validation scores. `random_state=42` makes the split reproducible.

In [6]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(gss.split(df, groups=df["lesion_id"]))

train_df = df.iloc[train_idx].reset_index(drop=True)
val_df   = df.iloc[val_idx].reset_index(drop=True)

## 7. Custom `SkinDataset`

A minimal PyTorch `Dataset` wrapper:
- `__len__` returns the number of samples so DataLoader knows when an epoch ends.
- `__getitem__` loads one JPEG on-the-fly, converts to RGB (handles grayscale/RGBA edge cases), applies the transform pipeline, and returns `(image_tensor, label_int)`.

Lazy loading keeps RAM usage low – images are never all in memory at once.

In [7]:
class SkinDataset(Dataset):
    def __init__(self, df, img_dir, transform):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = os.path.join(self.img_dir, f"{row['image']}.jpg")

        img = Image.open(path).convert("RGB")
        img = self.transform(img)

        return img, int(row["label_idx"])


## 8. Data Augmentation & Normalization

**Training transforms** (applied randomly each epoch):
- `RandomResizedCrop` – simulates varying zoom/framing of lesions.
- `RandomHorizontalFlip` – lesions have no left/right meaning; free augmentation.
- `ColorJitter` – mild perturbation of brightness/contrast/saturation/hue, helping against varying lighting and device calibration.

**Validation transforms** (deterministic):
- Resize then centre-crop – no randomness; ensures reproducible evaluation.

Both pipelines apply **ImageNet normalisation** (mean/std) because EfficientNet was pre-trained on ImageNet.

In [8]:
train_tfms = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.1, 0.1, 0.1, 0.05),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

val_tfms = transforms.Compose([
    transforms.Resize(IMG_SIZE + 32),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

## 9. DataLoaders & Class-Balanced Sampling

ISIC 2019 is heavily imbalanced (NV nevus dominates). Without correction, the model learns to predict the majority class.

**WeightedRandomSampler** fixes this by oversampling rare classes so each mini-batch sees a roughly uniform class distribution. `replacement=True` means a minority-class image can be drawn multiple times per epoch.

**Performance flags:**
- `pin_memory=True` – locks CPU tensors in RAM so GPU DMA transfers are faster.
- `persistent_workers=True` – keeps worker processes alive between epochs, avoiding fork overhead.

In [9]:
train_ds = SkinDataset(train_df, IMG_DIR, train_tfms)
val_ds   = SkinDataset(val_df, IMG_DIR, val_tfms)

class_counts = train_df["label_idx"].value_counts().sort_index()
class_weights = 1.0 / class_counts
sample_weights = train_df["label_idx"].map(class_weights).values

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True,
)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 6 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


## 10. EfficientNet-B0 with Custom Head

We load EfficientNet-B0 with ImageNet weights (**transfer learning**) and replace only the final linear layer to output 7 logits instead of 1000.

**Why EfficientNet?** It achieves strong accuracy with far fewer parameters than ResNet/VGG, making it fast to fine-tune on medical imaging datasets.

Freezing the backbone is not done here – all layers are trained, which works well when the dataset is moderately large (ISIC 2019 has ~25k images).

In [10]:
model = models.efficientnet_b0(weights="IMAGENET1K_V1")
model.classifier[1] = nn.Linear(
    model.classifier[1].in_features,
    len(classes),
)
model.to(DEVICE)

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth
100%|██████████| 20.5M/20.5M [00:00<00:00, 151MB/s]


EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

## 11. Loss, Optimiser, LR Schedule & AMP Scaler

- **CrossEntropyLoss** – standard for multi-class classification; expects raw logits.
- **AdamW** – Adam with decoupled weight decay; generally outperforms Adam on fine-tuning.
- **CosineAnnealingLR** – smoothly decays LR from `LR` to near-zero over `EPOCHS`, then could restart. Prevents getting stuck in sharp minima late in training.
- **GradScaler** – enables **Automatic Mixed Precision (AMP)**. Keeps weights in FP32 but does the forward/backward in FP16, roughly doubling throughput on modern GPUs with no accuracy loss.

In [11]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS
)

scaler = torch.cuda.amp.GradScaler()

/tmp/ipykernel_20/1621428337.py:8: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


## 12. Training Loop

Each epoch has two phases:

**Train phase:**
1. Move batch to GPU with `non_blocking=True` for overlapped CPU→GPU transfer.
2. `zero_grad(set_to_none=True)` – slightly faster than zeroing; sets gradients to `None` instead of 0.
3. `autocast()` block runs the forward pass in FP16.
4. `scaler.scale(loss).backward()` – scales loss to prevent FP16 underflow, then back-props.
5. `scaler.step` / `scaler.update` – unscales gradients and updates weights.

**Val phase:**
- `torch.no_grad()` disables gradient tracking (saves memory and speeds up inference).
- Collect all predictions, compute **macro F1** – the right metric for imbalanced datasets because it gives equal weight to each class regardless of frequency.

**Best-model checkpoint:** saved whenever macro F1 improves, so the final artefact is the best epoch, not the last.

In [12]:
best_macro_f1 = 0.0

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0

    for x, y in train_loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast():
            logits = model(x)
            loss = criterion(logits, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()

    scheduler.step()

    model.eval()
    preds, targets = [], []

    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(DEVICE)
            logits = model(x)
            preds.append(logits.argmax(1).cpu())
            targets.append(y)

    preds = torch.cat(preds).numpy()
    targets = torch.cat(targets).numpy()

    report = classification_report(
        targets, preds, target_names=classes, output_dict=True
    )

    macro_f1 = report["macro avg"]["f1-score"]

    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print(f"Train loss: {train_loss / len(train_loader):.4f}")
    print(f"Macro F1:   {macro_f1:.4f}")
    print(classification_report(targets, preds, target_names=classes, digits=3))

    if macro_f1 > best_macro_f1:
        best_macro_f1 = macro_f1
        torch.save(model.state_dict(), "best_model.pt")

/tmp/ipykernel_20/654277856.py:13: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():



Epoch 1/25
Train loss: 1.1645
Macro F1:   0.4719
              precision    recall  f1-score   support

          AK      0.217     0.412     0.284       165
         BCC      0.602     0.621     0.611       601
         BKL      0.412     0.462     0.436       489
          DF      0.141     0.543     0.224        35
         MEL      0.560     0.526     0.543       944
          NV      0.815     0.671     0.736      1937
        VASC      0.338     0.772     0.471        57

    accuracy                          0.597      4228
   macro avg      0.441     0.572     0.472      4228
weighted avg      0.646     0.597     0.615      4228



/tmp/ipykernel_20/654277856.py:13: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 6 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(



Epoch 2/25
Train loss: 0.8966
Macro F1:   0.5264
              precision    recall  f1-score   support

          AK      0.261     0.618     0.367       165
         BCC      0.553     0.627     0.588       601
         BKL      0.414     0.573     0.480       489
          DF      0.133     0.486     0.209        35
         MEL      0.633     0.467     0.537       944
          NV      0.844     0.695     0.762      1937
        VASC      0.729     0.754     0.741        57

    accuracy                          0.616      4228
   macro avg      0.509     0.603     0.526      4228
weighted avg      0.676     0.616     0.634      4228



/tmp/ipykernel_20/654277856.py:13: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 6 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


...


Epoch 23/25
Train loss: 0.2870
Macro F1:   0.5734
              precision    recall  f1-score   support

          AK      0.346     0.430     0.384       165
         BCC      0.649     0.717     0.681       601
         BKL      0.540     0.544     0.542       489
          DF      0.316     0.343     0.329        35
         MEL      0.639     0.552     0.592       944
          NV      0.803     0.821     0.812      1937
        VASC      0.938     0.526     0.674        57

    accuracy                          0.691      4228
   macro avg      0.604     0.562     0.573      4228
weighted avg      0.694     0.691     0.690      4228



/tmp/ipykernel_20/654277856.py:13: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 6 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(



Epoch 24/25
Train loss: 0.2881
Macro F1:   0.5731
              precision    recall  f1-score   support

          AK      0.381     0.418     0.399       165
         BCC      0.677     0.694     0.685       601
         BKL      0.519     0.550     0.534       489
          DF      0.364     0.343     0.353        35
         MEL      0.643     0.555     0.596       944
          NV      0.789     0.829     0.808      1937
        VASC      0.903     0.491     0.636        57

    accuracy                          0.692      4228
   macro avg      0.611     0.554     0.573      4228
weighted avg      0.691     0.692     0.690      4228



/tmp/ipykernel_20/654277856.py:13: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 6 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(



Epoch 25/25
Train loss: 0.2868
Macro F1:   0.5744
              precision    recall  f1-score   support

          AK      0.374     0.388     0.381       165
         BCC      0.655     0.740     0.695       601
         BKL      0.548     0.503     0.525       489
          DF      0.333     0.343     0.338        35
         MEL      0.622     0.581     0.601       944
          NV      0.800     0.816     0.808      1937
        VASC      0.842     0.561     0.674        57

    accuracy                          0.692      4228
   macro avg      0.596     0.562     0.574      4228
weighted avg      0.691     0.692     0.690      4228

